# Spark Performance Tuning & Adaptive Query Execution (AQE)

## Why Performance Tuning Matters

Spark jobs fail or run slowly for three primary reasons:

1. **Excessive shuffle** — Moving data across the network is the single most expensive operation in a distributed system. Every `groupBy`, `join`, `distinct`, and `repartition` can trigger a shuffle. The goal is to minimise the *volume* and *frequency* of shuffles.

2. **Data skew** — When one partition holds dramatically more data than others, that partition's task becomes the bottleneck (the "straggler"). The job cannot finish until the last task finishes, so even if 999 tasks complete in 30 seconds, one skewed task running for 10 minutes dominates.

3. **Spill to disk** — Spark operates on data in RAM. When a sort, aggregation, or join exhausts the executor's memory budget, it spills intermediate data to disk. Disk I/O is 100–1000x slower than RAM, so a single spill can turn a 2-minute job into a 20-minute job.

This notebook walks through the most impactful tuning levers in Spark 3.5, with emphasis on **Adaptive Query Execution (AQE)** — Spark's runtime self-optimisation framework introduced in Spark 3.0 and greatly improved through 3.5.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("PerformanceTuning_AQE")
    # --- AQE: the master switch ---
    # Default in Spark 3.2+ is True, but we set it explicitly for clarity.
    .config("spark.sql.adaptive.enabled", "true")
    # AQE feature 1: coalesce post-shuffle partitions that are too small
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    # Target size (bytes) for each coalesced partition
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64MB")
    # AQE feature 2: skew join handling
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    # A partition is "skewed" if it is > this factor × median partition size
    .config("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
    # AND the partition is larger than this absolute threshold
    .config("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256MB")
    # AQE feature 3: runtime switch from SortMergeJoin → BroadcastHashJoin
    .config("spark.sql.adaptive.autoBroadcastJoinThreshold", "10MB")
    # Number of shuffle partitions BEFORE AQE coalesces them
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"AQE enabled   : {spark.conf.get('spark.sql.adaptive.enabled')}")

## Adaptive Query Execution (AQE) — Deep Dive

Before AQE, Spark built a **static physical plan** at compile time using table statistics from the metastore (often stale or missing). AQE replaces this with a **re-optimisation loop**: after each shuffle stage completes, Spark re-examines the real runtime statistics and may rewrite the rest of the plan.

```
QUERY SUBMITTED
       │
       ▼
 ┌─────────────┐
 │ Parse & Analyse │
 └──────┬──────┘
        │
        ▼
 ┌─────────────┐
 │ Logical Plan │
 └──────┬──────┘
        │  Catalyst optimiser (rule-based + cost-based)
        ▼
 ┌──────────────────┐
 │ Initial Physical  │
 │      Plan         │
 └──────┬───────────┘
        │
        ▼
 ┌──────────────────────────────────────┐
 │         STAGE EXECUTION LOOP (AQE)   │
 │                                      │
 │  Execute stage ──► collect stats     │
 │          │                           │
 │          ▼                           │
 │  ┌───────────────────────────────┐   │
 │  │  Re-optimise remaining plan?  │   │
 │  │  1. Coalesce small partitions │   │
 │  │  2. Handle skewed partitions  │   │
 │  │  3. Switch join strategy      │   │
 │  └───────────────────────────────┘   │
 │          │                           │
 │          ▼                           │
 │  Next stage (updated plan)           │
 └──────────────────────────────────────┘
        │
        ▼
     RESULT
```

### The Three AQE Features

| Feature | Problem it solves | Mechanism |
|---|---|---|
| **Dynamic partition coalescing** | Too many tiny post-shuffle partitions → task overhead dominates | After a shuffle, AQE merges adjacent small partitions until each reaches `advisoryPartitionSizeInBytes` |
| **Skew join handling** | One fat partition makes the join 10–100x slower than the median task | AQE detects skewed partitions at runtime and splits them; it also replicates the matching side's data for those splits |
| **Dynamic join strategy switching** | Plan said SortMergeJoin because stats were missing; at runtime one side turns out to be tiny | AQE substitutes a BroadcastHashJoin for that stage, eliminating the sort entirely |

In [ ]:
# ── AQE Feature 1: Dynamic Partition Coalescing ──────────────────────────────
# We create a DataFrame with high cardinality values across many groups,
# perform a shuffle (groupBy), and observe how AQE shrinks partition count.

# Force 200 shuffle partitions BEFORE AQE does its work
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Build a moderately-sized DataFrame: 500,000 rows, ~50 distinct groups.
# After shuffle there will be at most 50 non-empty partitions out of 200.
# AQE will coalesce the many empty/tiny partitions into fewer, fuller ones.
df_large = spark.range(500_000).withColumn(
    "group_key", (F.col("id") % 50).cast("int")
).withColumn(
    "value", F.rand() * 1000
)

# Trigger a shuffle via groupBy + agg
agg_df = df_large.groupBy("group_key").agg(
    F.count("*").alias("cnt"),
    F.avg("value").alias("avg_value")
)

# Materialize so AQE can do its work
agg_df.cache()
agg_df.count()  # action — triggers the shuffle and AQE coalescing

# How many partitions does the result actually have?
actual_partitions = agg_df.rdd.getNumPartitions()
print(f"spark.sql.shuffle.partitions setting : 200")
print(f"Actual partitions after AQE coalescing: {actual_partitions}")
print("")
print("Why fewer? AQE observed that 200 post-shuffle partitions for only 50")
print("distinct groups produced ~150 empty or tiny partitions and merged them.")

agg_df.show(10)

In [ ]:
# ── Broadcast Join Threshold & Plan Inspection ───────────────────────────────
# spark.sql.autoBroadcastJoinThreshold controls the *static* broadcast decision
# made at plan time.  AQE's autoBroadcastJoinThreshold is the *runtime* version.
# Here we demonstrate both so the difference is clear.

# Build a large "fact" table and a small "dimension" table
facts = spark.range(1_000_000).withColumn(
    "category_id", (F.col("id") % 100).cast("int")
).withColumn("amount", F.rand() * 500)

dims = spark.range(100).withColumnRenamed("id", "category_id").withColumn(
    "category_name", F.concat(F.lit("cat_"), F.col("category_id").cast("string"))
)

# ── Scenario A: threshold = -1 → broadcast disabled → SortMergeJoin ──
print("=" * 60)
print("SCENARIO A: autoBroadcastJoinThreshold = -1 (disabled)")
print("=" * 60)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
# Also disable AQE's dynamic broadcast for a clean comparison
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", "-1")

join_no_broadcast = facts.join(dims, on="category_id", how="inner")
# explain() prints the physical plan; look for "SortMergeJoin"
join_no_broadcast.explain(mode="formatted")

print()
print("=" * 60)
print("SCENARIO B: autoBroadcastJoinThreshold = 10MB → BroadcastHashJoin")
print("=" * 60)

# Re-enable broadcast; dims table is well under 10 MB
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))  # 10 MB
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

join_broadcast = facts.join(dims, on="category_id", how="inner")
# Look for "BroadcastHashJoin" and "BroadcastExchange" in the plan
join_broadcast.explain(mode="formatted")

print()
print("KEY INSIGHT:")
print("  SortMergeJoin  → both sides sorted + merged   → 2 shuffles")
print("  BroadcastHashJoin → small side broadcast      → 0 shuffles")
print("  Eliminating the shuffle is often a 5–20x speedup for this pattern.")

In [ ]:
# ── Shuffle Partition Tuning: N=200 vs N=20 ──────────────────────────────────
# The right value for spark.sql.shuffle.partitions depends on your data volume.
#   Too many → thousands of tiny tasks, JVM scheduling overhead dominates.
#   Too few  → each task processes too much data, spill risk increases.
# Rule of thumb: aim for ~128–256 MB per post-shuffle partition.
# With AQE on, an initially-high value gets auto-coalesced downward;
# an initially-low value cannot be split upward (that's a different tuning problem).

# Disable AQE coalescing so we see the raw difference from partition count
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")

def run_shuffle_benchmark(n_partitions: int) -> float:
    """Run a groupBy+sort query with a specific shuffle partition count and return elapsed seconds."""
    spark.conf.set("spark.sql.shuffle.partitions", str(n_partitions))
    df = spark.range(2_000_000).withColumn("k", (F.col("id") % 500).cast("int"))
    t0 = time.time()
    # Two-stage shuffle: groupBy (stage 1), orderBy (stage 2)
    result = df.groupBy("k").agg(F.sum("id").alias("s")).orderBy("s", ascending=False)
    result.write.format("noop").mode("overwrite").save()  # materialize without writing files
    elapsed = time.time() - t0
    return elapsed

# Run with 200 partitions
t_200 = run_shuffle_benchmark(200)
print(f"shuffle.partitions=200  → {t_200:.2f}s")

# Run with 20 partitions (closer to the 500 distinct groups in our data)
t_20 = run_shuffle_benchmark(20)
print(f"shuffle.partitions=20   → {t_20:.2f}s")

print()
print(f"Ratio (200 / 20): {t_200 / t_20:.2f}x")
print()
print("With AQE on (the normal state), the 200-partition run would auto-coalesce")
print("and perform similarly to 20.  The benchmark above shows the *raw* impact.")

# Re-enable AQE coalescing for subsequent cells
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

## Key Tuning Knobs Reference

The table below covers the most frequently adjusted configuration properties. "Direction" means whether you typically increase or decrease the value to improve performance.

| Config property | Default | What it controls | Typical tuning direction |
|---|---|---|---|
| `spark.sql.adaptive.enabled` | `true` (3.2+) | Master switch for AQE | Always keep `true` |
| `spark.sql.adaptive.coalescePartitions.enabled` | `true` | Auto-merge small post-shuffle partitions | Keep `true`; tune `advisoryPartitionSizeInBytes` |
| `spark.sql.adaptive.advisoryPartitionSizeInBytes` | `64MB` | Target size for each coalesced partition | Increase to 128–256 MB for wide/complex rows |
| `spark.sql.adaptive.skewJoin.enabled` | `true` | Split skewed partitions in joins | Keep `true`; tune the factor/threshold |
| `spark.sql.adaptive.skewJoin.skewedPartitionFactor` | `5` | Multiplier above median to declare a partition skewed | Lower (e.g. 3) if you have persistent mild skew |
| `spark.sql.shuffle.partitions` | `200` | Initial number of reduce-side shuffle partitions | Start high (e.g. 1000) and let AQE coalesce; without AQE, tune to ~128 MB per partition |
| `spark.sql.autoBroadcastJoinThreshold` | `10MB` | Max size of a relation to auto-broadcast at plan time | Raise cautiously (e.g. 50 MB) if you have reliable stats |
| `spark.sql.adaptive.autoBroadcastJoinThreshold` | `10MB` | Same as above but decided at *runtime* by AQE | Usually more reliable than the static version |
| `spark.default.parallelism` | 2 × cores | Default partition count for RDD operations | Match to your cluster core count |
| `spark.executor.memory` | `1g` | JVM heap per executor | Increase until spill stops; balance with `cores` |
| `spark.memory.fraction` | `0.6` | Fraction of heap used for execution + storage | Lower to give more room to user code objects |
| `spark.memory.storageFraction` | `0.5` | Fraction of the above reserved for cached data | Increase if you cache large DataFrames |
| `spark.serializer` | Java | Serialization library for shuffle data | Switch to `org.apache.spark.serializer.KryoSerializer` for 2–5x speedup |

In [ ]:
# ── Partition Pruning ─────────────────────────────────────────────────────────
# Partition pruning lets Spark skip entire directories on disk when a filter
# exactly matches the partition column.  This is purely a *read-time* optimisation
# and requires the data to be written with partitionBy().
#
# Without pruning:  scan every file → filter in memory → discard non-matching rows
# With pruning:     skip non-matching directories → read only relevant files

import os

DATA_DIR = "/home/jovyan/data"
PARTITIONED_PATH = f"{DATA_DIR}/sales_partitioned"

# Step 1: Write partitioned data
sales = (
    spark.range(200_000)
    .withColumn("year",  (2020 + (F.col("id") % 4)).cast("int"))   # years 2020–2023
    .withColumn("month", (1    + (F.col("id") % 12)).cast("int"))  # months 1–12
    .withColumn("amount", F.rand() * 1000)
)

(
    sales
    .write
    .partitionBy("year", "month")   # creates directory tree: year=2020/month=1/...
    .mode("overwrite")
    .parquet(PARTITIONED_PATH)
)
print(f"Wrote partitioned data to {PARTITIONED_PATH}")

# Step 2: Read back with a filter on the partition key
sales_read = spark.read.parquet(PARTITIONED_PATH)

# Filter on the partition column — Spark should push this to storage layer
filtered = sales_read.filter((F.col("year") == 2022) & (F.col("month") == 6))

print()
print("Physical plan (look for 'PartitionFilters' — confirms pruning is active):")
print("=" * 70)
filtered.explain(mode="simple")

# Show row count to confirm only year=2022/month=6 rows are returned
count = filtered.count()
print(f"Row count for year=2022, month=6: {count}")
print()
print("INSIGHT: Without partitioning, reading 200,000 rows and filtering in RAM.")
print("With partitioning:  1/(4 years × 12 months) = ~1/48 of data read from disk.")

In [ ]:
# Clean up and stop the session
spark.catalog.clearCache()
spark.stop()
print("SparkSession stopped. Notebook complete.")